In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
products= pd.read_csv("products.CSV")
customers =pd.read_csv("customers.CSV")
sales=pd.read_csv("sales.CSV")

In [3]:
#creating back up files
products_backup=products.copy()
customers_backup=customers.copy()
sales_backup=sales.copy()

In [4]:

products.head()

,product_key,product_id,product_number,product_name,category_id,category_name,sub_category,maintenance,cost,product_line,start_date
0,1,210,FR-R92B-58,HL Road Frame - Black- 58,CO_RF,Components,Road Frames,Yes,0,Road,2003-07-01
1,2,211,FR-R92R-58,HL Road Frame - Red- 58,CO_RF,Components,Road Frames,Yes,0,Road,2003-07-01
2,3,348,BK-M82B-38,Mountain-100 Black- 38,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01
3,4,349,BK-M82B-42,Mountain-100 Black- 42,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01
4,5,350,BK-M82B-44,Mountain-100 Black- 44,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01


In [5]:
customers.head()

,customer_key,customer_id,customer_number,first_name,last_name,country,marital_status,gender,birth_date,create_date
0,1,11000,AW00011000,Jon,Yang,Australia,Married,Male,1971-10-06,2025-10-06
1,2,11001,AW00011001,Eugene,Huang,Australia,Single,Male,1976-05-10,2025-10-06
2,3,11002,AW00011002,Ruben,Torres,Australia,Married,Male,1971-02-09,2025-10-06
3,4,11003,AW00011003,Christy,Zhu,Australia,Single,Female,1973-08-14,2025-10-06
4,5,11004,AW00011004,Elizabeth,Johnson,Australia,Single,Female,1979-08-05,2025-10-06


In [6]:

sales.head()

,order_number,product_key,customer_key,order_date,shipping_date,due_date,sales_amount,quantity,price
0,SO43697,20,10769,2010-12-29,2011-01-05,2011-01-10,3578,1,3578
1,SO43698,9,17390,2010-12-29,2011-01-05,2011-01-10,3400,1,3400
2,SO43699,9,14864,2010-12-29,2011-01-05,2011-01-10,3400,1,3400
3,SO43700,41,3502,2010-12-29,2011-01-05,2011-01-10,699,1,699
4,SO43701,9,4,2010-12-29,2011-01-05,2011-01-10,3400,1,3400


In [7]:
# replace N/A to null values
sales.replace("N/A", pd.NA, inplace=True)
products.replace("N/A", pd.NA, inplace=True)
customers.replace("N/A", pd.NA, inplace=True)

In [8]:
sales.isna().sum()

,0
order_number,0
product_key,0
customer_key,0
order_date,19
shipping_date,0
due_date,0
sales_amount,0
quantity,0
price,0


In [9]:
products.isna().sum()

,0
product_key,0
product_id,0
product_number,0
product_name,0
category_id,0
category_name,7
sub_category,7
maintenance,7
cost,0
product_line,17


In [10]:

customers.isna().sum()

,0
customer_key,0
customer_id,0
customer_number,0
first_name,0
last_name,0
country,337
marital_status,0
gender,32
birth_date,36
create_date,0


In [11]:
#drop duplicates
sales = sales.drop_duplicates()
products = products.drop_duplicates()
customers = customers.drop_duplicates()

In [12]:
#cleaning sales table
#convert to datetime
sales['order_date'] = pd.to_datetime(sales['order_date'])
sales['shipping_date'] = pd.to_datetime(sales['shipping_date'])
sales['due_date']=pd.to_datetime(sales['due_date'])

In [13]:
#missing order date for each customer_key
sales.loc[sales["order_date"].isna(), ["customer_key"]]


,customer_key
35319,11820
35320,5323
35321,5323
35322,10522
35323,10522
35420,1374
35422,1374
35427,2009
35432,10996
36046,15792


In [14]:
sales["days_to_shipping_start"] = (
    sales["shipping_date"] - sales["order_date"]
).dt.days

In [15]:
avg_days = sales.groupby("customer_key")["days_to_shipping_start"].mean()

In [16]:
sales_copy = sales.copy()

In [17]:
estimated_order_date = (
    sales["shipping_date"]
    - pd.to_timedelta(sales["customer_key"].map(avg_days), unit="D")
)

In [18]:
sales.isna().sum()

,0
order_number,0
product_key,0
customer_key,0
order_date,19
shipping_date,0
due_date,0
sales_amount,0
quantity,0
price,0
days_to_shipping_start,19


In [19]:
sales["order_date"] = sales["order_date"].fillna(estimated_order_date)

We still got 3 missing order_date its occurs because the customer only appear where the order date is null

In [20]:

#we use overall missing date to fill those 3
global_avg = sales["days_to_shipping_start"].mean()

In [21]:
mask = sales["order_date"].isna()

sales.loc[mask,"order_date"] = (
    sales.loc[mask,"shipping_date"] -
    pd.to_timedelta(global_avg, unit="D")
)

In [22]:
sales["order_date"].isna().sum()

np.int64(0)

In [23]:
#cleaning the product table
products.isna().sum()

,0
product_key,0
product_id,0
product_number,0
product_name,0
category_id,0
category_name,7
sub_category,7
maintenance,7
cost,0
product_line,17


In [24]:
products["category_name"].unique()

array(['Components', 'Bikes', 'Clothing', 'Accessories', nan],
      dtype=object)

In [25]:
products["sub_category"].unique()

array(['Road Frames', 'Mountain Bikes', 'Road Bikes', 'Mountain Frames',
       'Socks', 'Forks', 'Wheels', 'Gloves', 'Headsets', 'Locks',
       'Lights', 'Panniers', 'Pumps', 'Bib-Shorts', 'Shorts', 'Tights',
       'Bottom Brackets', 'Bottles and Cages', 'Touring Bikes', 'Caps',
       'Chains', 'Cleaners', 'Cranksets', 'Brakes', 'Derailleurs',
       'Fenders', 'Touring Frames', 'Handlebars', 'Helmets',
       'Hydration Packs', 'Jerseys', nan, 'Tires and Tubes', 'Bike Racks',
       'Saddles', 'Bike Stands', 'Vests'], dtype=object)

In [27]:
products["product_line"].unique()

array(['Road', 'Mountain', nan, 'Touring', 'Other Sales'], dtype=object)

In [51]:
products[["product_number","product_name","category_id","category_name","sub_category","maintenance","product_line"]].drop_duplicates()

,product_number,product_name,category_id,category_name,sub_category,maintenance,product_line
0,FR-R92B-58,HL Road Frame - Black- 58,CO_RF,Components,Road Frames,Yes,Road
1,FR-R92R-58,HL Road Frame - Red- 58,CO_RF,Components,Road Frames,Yes,Road
2,BK-M82B-38,Mountain-100 Black- 38,BI_MB,Bikes,Mountain Bikes,Yes,Mountain
3,BK-M82B-42,Mountain-100 Black- 42,BI_MB,Bikes,Mountain Bikes,Yes,Mountain
4,BK-M82B-44,Mountain-100 Black- 44,BI_MB,Bikes,Mountain Bikes,Yes,Mountain
...,...,...,...,...,...,...,...
290,TT-T092,Touring Tire Tube,AC_TT,Accessories,Tires and Tubes,Yes,Touring
291,VE-C304-L,Classic Vest- L,CL_VE,Clothing,Vests,No,Other Sales
292,VE-C304-M,Classic Vest- M,CL_VE,Clothing,Vests,No,Other Sales
293,VE-C304-S,Classic Vest- S,CL_VE,Clothing,Vests,No,Other Sales


CO_PE category_id dont have category_name,sub_category_ and maintenance


In [28]:
products[
    products[["category_name","sub_category","maintenance"]].isna().all(axis=1)
][["product_name","category_id","category_name","sub_category","maintenance"]].drop_duplicates()

,product_name,category_id,category_name,sub_category,maintenance
251,LL Mountain Pedal,CO_PE,NaN,NaN,NaN
252,ML Mountain Pedal,CO_PE,NaN,NaN,NaN
253,HL Mountain Pedal,CO_PE,NaN,NaN,NaN
254,LL Road Pedal,CO_PE,NaN,NaN,NaN
255,ML Road Pedal,CO_PE,NaN,NaN,NaN
256,HL Road Pedal,CO_PE,NaN,NaN,NaN
257,Touring Pedal,CO_PE,NaN,NaN,NaN


By comparing this table with above table we can see this products are belongs to "components" category_name .

And also while they  assigning to sub_category we can consider the id code pattern , eg: CO_BB	Components	Bottom Brackets .
Here in our category_id CO_PE we have components "pedals".

Finally we have to consider the maintenance and product_line lets fix them


In [29]:
products[
    products[["category_name","sub_category","maintenance","product_line"]].isna().all(axis=1)
][["product_name","category_id","category_name","sub_category","maintenance","product_line"]].drop_duplicates()

,product_name,category_id,category_name,sub_category,maintenance,product_line


We can  see that there are no missing values for prodcut_line with  missing category_name,sub_category,maintenance. There is no relationship between product line with category_name and sub_category.


Lets fix the maintenance

In [35]:
#Check whether maintenance and the product_line has relationship
pd.crosstab(
    products[products["category_name"]=="Components"]["product_line"],
    products[products["category_name"]=="Components"]["maintenance"]
)

maintenance,No,Yes
product_line,,
Mountain,6,34
Road,5,39
Touring,6,20


Most of the maintenance are yes in  components category. But this is not 100% sure . We can ask from managers to fill them or we can use another approach using common sense.
Most mechanical bike components require maintenance, especially drivetrain parts. So the maintenance is "Yes".


In [36]:
products[products["category_name"] == "Components"][
    ["sub_category","maintenance"]
].drop_duplicates().sort_values("sub_category")

,sub_category,maintenance
100,Bottom Brackets,Yes
171,Brakes,Yes
166,Chains,Yes
168,Cranksets,Yes
172,Derailleurs,Yes
50,Forks,Yes
235,Handlebars,No
73,Headsets,No
20,Mountain Frames,Yes
0,Road Frames,Yes


In [37]:
#lets update missing rows in missing cateogeory details
products.loc[products["category_id"] == "CO_PE",
             ["category_name","sub_category","maintenance"]] = ["Components","Pedals","Yes"]

In [38]:
#check
products[products["category_id"] == "CO_PE"]

,product_key,product_id,product_number,product_name,category_id,category_name,sub_category,maintenance,cost,product_line,start_date
251,252,542,PD-M282,LL Mountain Pedal,CO_PE,Components,Pedals,Yes,18,Mountain,2013-07-01
252,253,543,PD-M340,ML Mountain Pedal,CO_PE,Components,Pedals,Yes,28,Mountain,2013-07-01
253,254,544,PD-M562,HL Mountain Pedal,CO_PE,Components,Pedals,Yes,36,Mountain,2013-07-01
254,255,545,PD-R347,LL Road Pedal,CO_PE,Components,Pedals,Yes,18,Road,2013-07-01
255,256,546,PD-R563,ML Road Pedal,CO_PE,Components,Pedals,Yes,28,Road,2013-07-01
256,257,547,PD-R853,HL Road Pedal,CO_PE,Components,Pedals,Yes,36,Road,2013-07-01
257,258,548,PD-T852,Touring Pedal,CO_PE,Components,Pedals,Yes,36,Touring,2013-07-01


In [39]:
#lets check for missing values
products.isna().sum()

,0
product_key,0
product_id,0
product_number,0
product_name,0
category_id,0
category_name,0
sub_category,0
maintenance,0
cost,0
product_line,17


In [52]:
products[products["product_line"].isna()][["product_number","product_name","category_id","category_name","product_line"]]

,product_number,product_name,category_id,category_name,product_line
50,FK-1639,LL Fork,CO_FO,Components,NaN
51,FK-5136,ML Fork,CO_FO,Components,NaN
52,FK-9939,HL Fork,CO_FO,Components,NaN
73,HS-0296,LL Headset,CO_HS,Components,NaN
74,HS-2451,ML Headset,CO_HS,Components,NaN
75,HS-3479,HL Headset,CO_HS,Components,NaN
100,BB-7421,LL Bottom Bracket,CO_BB,Components,NaN
101,BB-8107,ML Bottom Bracket,CO_BB,Components,NaN
102,BB-9108,HL Bottom Bracket,CO_BB,Components,NaN
166,CH-0234,Chain,CO_CH,Components,NaN


We can see some Details in product_number and product_name which have kind of relationship to the product_line.
eg : (TT-T092,Touring Tire Tube,AC_TT,Accessories,Tires and Tubes,Yes  Touring )

This row contains product_number, product_name, category_id, category_name,sub_category, maintenance and product_line respectively.

We can think this as a logic:
* Consider product_number TT-T092 : the first letter in last
4 letters (4th element) , "T"  indicates the product_line ,which is "Touring".

* Consider product name "Touring Tire Tube" : We can see that the word "Touring" is consists in this product_name which also can be indicates the product_line "Touring".


By considering those two logics we can conclude that  :
* If the letter of 4th element in product number is one of ("T" , "M","R")   Then the Product_line is ("Touring" , "Mountain" ,"Road") Respectively and otherwise "Other Sales".

 OR

* If there is a word in product_name one of ("Touring" , "Mountain" ,   "Road") then the product_line is one of those.

or both.

No we are going to impute those missing product_lines using above logics and some common sense..


In [55]:
# lets check how accuracy is our logics
def validate_product_line_rule(products):
    code = products["product_number"].str[-4]
    rule1 = code.map({
        "T": "Touring",
        "M": "Mountain",
        "R": "Road"
    })
    rule2 = np.select(
        [
            products["product_name"].str.contains("Touring", case=False, na=False),
            products["product_name"].str.contains("Mountain", case=False, na=False),
            products["product_name"].str.contains("Road", case=False, na=False)
        ],
        ["Touring", "Mountain", "Road"],
        default=None
    )
    products["predicted_product_line"] = rule1.fillna(pd.Series(rule2)).fillna("Other Sales")
    check = products[products["product_line"].notna()].copy()
    mismatches = check[check["product_line"] != check["predicted_product_line"]]
    return mismatches

In [56]:
mismatches = validate_product_line_rule(products)
print(mismatches)

     product_key  product_id product_number                  product_name  \
70            71         470      GL-F110-L         Full-Finger Gloves- L   
71            72         469      GL-F110-M         Full-Finger Gloves- M   
72            73         468      GL-F110-S         Full-Finger Gloves- S   
77            78         451        LT-H902        Headlights - Dual-Beam   
78            79         452        LT-H903     Headlights - Weatherproof   
79            80         450        LT-T990  Taillights - Battery-Powered   
266          267         519        SE-R908           ML Road Seat/Saddle   
278          279         482      SO-R809-L               Racing Socks- L   
279          280         481      SO-R809-M               Racing Socks- M   
280          281         486        ST-1401        All-Purpose Bike Stand   

    category_id category_name sub_category maintenance  cost product_line  \
70        CL_GL      Clothing       Gloves          No    16     Mountain  

In [57]:
#check the accuracy
check = products[products["product_line"].notna()]
accuracy = (check["product_line"] == check["predicted_product_line"]).mean()
print("Rule accuracy:", accuracy)

Rule accuracy: 0.9640287769784173


We can see our logic is almost correct.!


In [58]:
#lets fill the missing values in product_line
def predict_product_line(row):
    # Rule 1 : product_number
    try:
        char = row["product_number"][3]

        if char == "T":
            return "Touring"
        elif char == "M":
            return "Mountain"
        elif char == "R":
            return "Road"
    except:
        pass

    # Rule 2 : product_name
    name = str(row["product_name"])

    if "Touring" in name:
        return "Touring"
    elif "Mountain" in name:
        return "Mountain"
    elif "Road" in name:
        return "Road"
    return "Other Sales"

In [59]:
#apply
products.loc[products["product_line"].isna(),"product_line"] = products.loc[products["product_line"].isna()].apply(predict_product_line, axis=1)

In [67]:
#verify
products["product_line"].isna().sum()

np.int64(0)

In [62]:
products.head()

,product_key,product_id,product_number,product_name,category_id,category_name,sub_category,maintenance,cost,product_line,start_date,predicted_product_line
0,1,210,FR-R92B-58,HL Road Frame - Black- 58,CO_RF,Components,Road Frames,Yes,0,Road,2003-07-01,Road
1,2,211,FR-R92R-58,HL Road Frame - Red- 58,CO_RF,Components,Road Frames,Yes,0,Road,2003-07-01,Road
2,3,348,BK-M82B-38,Mountain-100 Black- 38,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01,Mountain
3,4,349,BK-M82B-42,Mountain-100 Black- 42,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01,Mountain
4,5,350,BK-M82B-44,Mountain-100 Black- 44,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01,Mountain


In [69]:
#just check
products[products['product_number']=="FK-1639"]

,product_key,product_id,product_number,product_name,category_id,category_name,sub_category,maintenance,cost,product_line,start_date
50,51,391,FK-1639,LL Fork,CO_FO,Components,Forks,Yes,66,Other Sales,2012-07-01


We have imputed missing values.


In [109]:
products.head()

,product_key,product_id,product_number,product_name,category_id,category_name,sub_category,maintenance,cost,product_line,start_date
0,1,210,FR-R92B-58,HL Road Frame - Black- 58,CO_RF,Components,Road Frames,Yes,0,Road,2003-07-01
1,2,211,FR-R92R-58,HL Road Frame - Red- 58,CO_RF,Components,Road Frames,Yes,0,Road,2003-07-01
2,3,348,BK-M82B-38,Mountain-100 Black- 38,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01
3,4,349,BK-M82B-42,Mountain-100 Black- 42,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01
4,5,350,BK-M82B-44,Mountain-100 Black- 44,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01


We can see there are two  0 costs in same product.
We are going to use the median imputation with  which is calcuated by using median cost for " High Level Road Frame belongs to "Road Frame" subcategroy and "Components" category.

In [110]:

mask = (
    products["product_name"].str.startswith("HL") &
    (products["sub_category"] == "Road Frames") &
    (products["category_name"] == "Components")
)
median_cost = products.loc[mask & (products["cost"] > 0), "cost"].median()
products.loc[mask & (products["cost"] == 0), "cost"] = median_cost

In [128]:
#Check if product_number + start_date is unique
products.groupby(["product_number","start_date"]).size().reset_index(name="count").query("count > 1")

,product_number,start_date,count


In [129]:
#Check if same product has multiple costs on the same start_date
products.groupby(["product_number","start_date"])["cost"].nunique().reset_index(name="cost_count").query("cost_count > 1")

,product_number,start_date,cost_count


In [130]:
#Check if a product has multiple costs overall
products.groupby("product_number")["cost"].nunique().sort_values(ascending=False)

,cost
product_number,
WB-H098,1
BB-7421,1
BB-8107,1
BB-9108,1
BC-M005,1
...,...
BK-M18S-42,1
BK-M18S-40,1
BK-M18B-52,1


This can be happen because over the time the cost of same product can be change.

In [131]:
#check whether product_id is unique
products["product_id"].duplicated().sum()

np.int64(0)

In [74]:
#lets clean the customers table
customers

,customer_key,customer_id,customer_number,first_name,last_name,country,marital_status,gender,birth_date,create_date
0,1,11000,AW00011000,Jon,Yang,Australia,Married,Male,1971-10-06,2025-10-06
1,2,11001,AW00011001,Eugene,Huang,Australia,Single,Male,1976-05-10,2025-10-06
2,3,11002,AW00011002,Ruben,Torres,Australia,Married,Male,1971-02-09,2025-10-06
3,4,11003,AW00011003,Christy,Zhu,Australia,Single,Female,1973-08-14,2025-10-06
4,5,11004,AW00011004,Elizabeth,Johnson,Australia,Single,Female,1979-08-05,2025-10-06
...,...,...,...,...,...,...,...,...,...,...
18479,18480,29479,AW00029479,Tommy,Tang,France,Married,NaN,1969-06-30,2026-01-25
18480,18481,29480,AW00029480,Nina,Raji,United Kingdom,Single,NaN,1977-05-06,2026-01-25
18481,18482,29481,AW00029481,Ivan,Suri,Germany,Single,NaN,1965-07-04,2026-01-25
18482,18483,29482,AW00029482,Clayton,Zhang,France,Married,NaN,1964-09-01,2026-01-25


In [ ]:
del customers

In [88]:
customers_backup.isna().sum()

,0
customer_key,0
customer_id,0
customer_number,0
first_name,0
last_name,0
country,337
marital_status,0
gender,32
birth_date,36
create_date,0


In [89]:
customers_b2 = customers_backup.copy() #In case we want

In [90]:
customers_backup["birth_date"] = pd.to_datetime(customers_backup["birth_date"])

In [92]:
#calculate age using brithdate
customers_backup["age"] = (pd.Timestamp.today() - customers_backup["birth_date"]).dt.days // 365

In [93]:
#we use hierarchical imputation
#step 1 gender+ marital_status+ age
customers_backup["age"] = customers_backup["age"].fillna(
    customers_backup.groupby(["country","marital_status","gender"])["age"].transform("median")
)

In [94]:
#step 2 country + marital_status
customers_backup["age"] = customers_backup["age"].fillna(
    customers_backup.groupby(["country","marital_status"])["age"].transform("median")
)

In [95]:
#step 3 country only
customers_backup["age"] = customers_backup["age"].fillna(
    customers_backup.groupby("country")["age"].transform("median")
)

In [96]:
customers_backup.isna().sum()

,0
customer_key,0
customer_id,0
customer_number,0
first_name,0
last_name,0
country,337
marital_status,0
gender,32
birth_date,36
create_date,0


We could fil the all the missing values in age by using those steps. We dont need use overall median then.

In [98]:
#For now we keep the unknown countries and gender as unknown
customers_backup["country"] = customers_backup["country"].fillna("Unknown")
customers_backup["gender"] = customers_backup["gender"].fillna("Unknown")

In [99]:
customers_backup.isna().sum()

,0
customer_key,0
customer_id,0
customer_number,0
first_name,0
last_name,0
country,0
marital_status,0
gender,0
birth_date,36
create_date,0


Well finally we have cleaned all the data frames now we have the final touch

In [100]:
sales.head()

,order_number,product_key,customer_key,order_date,shipping_date,due_date,sales_amount,quantity,price,days_to_shipping_start
0,SO43697,20,10769,2010-12-29,2011-01-05,2011-01-10,3578,1,3578,7.0
1,SO43698,9,17390,2010-12-29,2011-01-05,2011-01-10,3400,1,3400,7.0
2,SO43699,9,14864,2010-12-29,2011-01-05,2011-01-10,3400,1,3400,7.0
3,SO43700,41,3502,2010-12-29,2011-01-05,2011-01-10,699,1,699,7.0
4,SO43701,9,4,2010-12-29,2011-01-05,2011-01-10,3400,1,3400,7.0


In [113]:
products.head()

,product_key,product_id,product_number,product_name,category_id,category_name,sub_category,maintenance,cost,product_line,start_date
0,1,210,FR-R92B-58,HL Road Frame - Black- 58,CO_RF,Components,Road Frames,Yes,869,Road,2003-07-01
1,2,211,FR-R92R-58,HL Road Frame - Red- 58,CO_RF,Components,Road Frames,Yes,869,Road,2003-07-01
2,3,348,BK-M82B-38,Mountain-100 Black- 38,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01
3,4,349,BK-M82B-42,Mountain-100 Black- 42,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01
4,5,350,BK-M82B-44,Mountain-100 Black- 44,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01


In [115]:
customers_backup.head()

,customer_key,customer_id,customer_number,first_name,last_name,country,marital_status,gender,birth_date,create_date,age
0,1,11000,AW00011000,Jon,Yang,Australia,Married,Male,1971-10-06,2025-10-06,54.0
1,2,11001,AW00011001,Eugene,Huang,Australia,Single,Male,1976-05-10,2025-10-06,49.0
2,3,11002,AW00011002,Ruben,Torres,Australia,Married,Male,1971-02-09,2025-10-06,55.0
3,4,11003,AW00011003,Christy,Zhu,Australia,Single,Female,1973-08-14,2025-10-06,52.0
4,5,11004,AW00011004,Elizabeth,Johnson,Australia,Single,Female,1979-08-05,2025-10-06,46.0


In [116]:
#drop column days_to_shipping_start from sales
sales.drop(columns=["days_to_shipping_start"], inplace=True)
#drop column birth_date from customers_backup
customers_backup.drop(columns=["birth_date"], inplace=True)

In [123]:
#rename  all the files as _cleaned
sales_cleaned = sales.copy()
products_cleaned = products.copy()
customers_cleaned = customers_backup.copy()


Lets save these files to further analysis


In [124]:
customers_cleaned.to_csv("customers_cleaned.csv", index=False)
products_cleaned.to_csv("products_cleaned.csv", index=False)
sales_cleaned.to_csv("sales_cleaned.csv", index=False)

In [125]:
import os
os.listdir()

['.config',
 'sales.CSV',
 'products.CSV',
 'products_cleaned.csv',
 'customers_cleaned.csv',
 'sales_cleaned.csv',
 'customers.CSV',
 'sample_data']